# ChaosOps Colab Training (Rebuilt RL Loop)

This notebook trains the rebuilt ChaosOps policy-gradient pipeline in Google Colab with memory-safe defaults.

What this notebook does:
- Installs required libraries
- Runs a smoke train (2-4 updates)
- Runs full training
- Checks reward variance and action diversity gates
- Saves artifacts

## 1) Runtime + Paths

Use `Runtime -> Change runtime type -> T4 GPU` before running.

Set `REPO_DIR` to your project path in Colab (for example `/content/Chaosops`).

In [ ]:
import os
import json
import subprocess
from pathlib import Path

try:
    import torch
except Exception:
    torch = None

REPO_DIR = Path('/content/Chaosops')
SMOKE_OUT = REPO_DIR / 'chaosops-rl-smoke-colab'
FULL_OUT = REPO_DIR / 'chaosops-qwen-grpo-colab'

print('Repo dir:', REPO_DIR)
if torch is not None:
    print('Torch version:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

## 2) Install Dependencies

This installs 4-bit and RL training dependencies needed by `chaosops/train.py`.

In [ ]:
%pip install -U pip
%pip install -U "torch" "transformers>=4.44.0" "datasets" "trl" "accelerate" "peft" "bitsandbytes"

# Optional: if your run expects these from project requirements
%pip install -U fastapi uvicorn pydantic

## 3) Ensure Repository Exists

If your repo is not already at `/content/Chaosops`, clone or copy it first.

In [ ]:
if not REPO_DIR.exists():
    raise FileNotFoundError(
        'REPO_DIR not found. Clone your repo first, for example:\n'
        '!git clone <your_repo_url> /content/Chaosops'
    )

train_file = REPO_DIR / 'chaosops' / 'train.py'
if not train_file.exists():
    raise FileNotFoundError(f'Missing training script: {train_file}')

print('Training script found:', train_file)

## 4) Smoke Train (3 Updates)

This validates the run path and checks reward variance + action diversity behavior early.

In [ ]:
cmd = [
    'python', str(REPO_DIR / 'chaosops' / 'train.py'),
    '--train_steps', '3',
    '--group_size', '2',
    '--max_new_tokens', '32',
    '--output_dir', str(SMOKE_OUT),
]

print('Running:', ' '.join(cmd))
proc = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f'Smoke run failed with code {proc.returncode}')

## 5) Inspect Smoke Metrics and Gates

Checks:
- `reward_variance > 0`
- top action repeat ratio <= 0.70
- non-constant loss

In [ ]:
metrics_path = SMOKE_OUT / 'train_metrics.json'
if not metrics_path.exists():
    raise FileNotFoundError(f'Missing metrics file: {metrics_path}')

with open(metrics_path, 'r', encoding='utf-8') as f:
    smoke_metrics = json.load(f)

print('Smoke updates:', len(smoke_metrics))
for row in smoke_metrics:
    actions = row.get('action_distribution', {})
    total_actions = max(1, sum(actions.values()))
    top_action = max(actions.values()) if actions else 0
    repeat_ratio = top_action / total_actions
    print({
        'update': row.get('update'),
        'avg_reward': row.get('avg_reward'),
        'reward_variance': row.get('reward_variance'),
        'loss': row.get('loss'),
        'repeat_ratio': round(repeat_ratio, 3),
        'success_rate': row.get('success_rate'),
    })

# Gate checks
assert any(r.get('reward_variance', 0.0) > 1e-8 for r in smoke_metrics), 'Reward variance gate failed'
assert len({round(float(r.get('loss', 0.0)), 8) for r in smoke_metrics}) > 1, 'Loss did not change'
print('Smoke gate checks passed.')

## 6) Full Training Run

After smoke passes, run a longer training session.

In [ ]:
full_steps = 24

cmd = [
    'python', str(REPO_DIR / 'chaosops' / 'train.py'),
    '--train_steps', str(full_steps),
    '--group_size', '2',
    '--max_new_tokens', '32',
    '--learning_rate', '2e-4',
    '--gamma', '0.99',
    '--temperature', '0.8',
    '--top_p', '0.9',
    '--epsilon_random', '0.20',
    '--output_dir', str(FULL_OUT),
]

print('Running:', ' '.join(cmd))
proc = subprocess.run(cmd, cwd=str(REPO_DIR), text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f'Full training failed with code {proc.returncode}')

## 7) Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

full_metrics_path = FULL_OUT / 'train_metrics.json'
if not full_metrics_path.exists():
    raise FileNotFoundError(f'Missing full metrics file: {full_metrics_path}')

with open(full_metrics_path, 'r', encoding='utf-8') as f:
    full_metrics = json.load(f)

updates = [m['update'] for m in full_metrics]
avg_rewards = [m['avg_reward'] for m in full_metrics]
losses = [m['loss'] for m in full_metrics]
success = [m['success_rate'] for m in full_metrics]
reward_var = [m['reward_variance'] for m in full_metrics]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(updates, avg_rewards)
axes[0, 0].set_title('Avg Reward')
axes[0, 1].plot(updates, losses)
axes[0, 1].set_title('Loss')
axes[1, 0].plot(updates, success)
axes[1, 0].set_title('Success Rate')
axes[1, 1].plot(updates, reward_var)
axes[1, 1].set_title('Reward Variance')
for ax in axes.flat:
    ax.set_xlabel('Update')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 8) Validation Summary

This checks the same required signals after full training.

In [ ]:
def max_repeat_ratio(action_distribution):
    total = max(1, sum(action_distribution.values()))
    top = max(action_distribution.values()) if action_distribution else 0
    return top / total

repeat_ratios = [max_repeat_ratio(m.get('action_distribution', {})) for m in full_metrics]
loss_unique = len({round(float(m.get('loss', 0.0)), 8) for m in full_metrics})
reward_var_positive = any(float(m.get('reward_variance', 0.0)) > 1e-8 for m in full_metrics)
success_improved = max(float(m.get('success_rate', 0.0)) for m in full_metrics) > 0.0

summary = {
    'max_repeat_ratio': max(repeat_ratios) if repeat_ratios else 0.0,
    'reward_variance_nonconstant': reward_var_positive,
    'loss_changes': loss_unique > 1,
    'success_rate_above_zero': success_improved,
}
print(json.dumps(summary, indent=2))

assert summary['max_repeat_ratio'] <= 0.70, 'Action diversity criterion failed'
assert summary['reward_variance_nonconstant'], 'Reward variance criterion failed'
assert summary['loss_changes'], 'Loss-change criterion failed'
assert summary['success_rate_above_zero'], 'Success-rate criterion failed'
print('Validation checks passed.')

## 9) Optional: Run Policy Eval Script

In [ ]:
eval_cmd = [
    'python', str(REPO_DIR / 'chaosops' / 'eval.py'),
    '--adapter_dir', str(FULL_OUT),
    '--episodes', '16',
]
print('Running:', ' '.join(eval_cmd))
proc = subprocess.run(eval_cmd, cwd=str(REPO_DIR), text=True, capture_output=True)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f'Eval failed with code {proc.returncode}')

## 10) Optional: Export Trained Artifacts

In [ ]:
archive_path = REPO_DIR / 'chaosops_qwen_grpo_colab.tar.gz'

if FULL_OUT.exists():
    tar_cmd = [
        'tar', '-czf', str(archive_path), '-C', str(REPO_DIR), FULL_OUT.name
    ]
    proc = subprocess.run(tar_cmd, text=True, capture_output=True)
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f'Archive failed with code {proc.returncode}')
    print('Saved archive:', archive_path)
else:
    print('No trained output directory found at', FULL_OUT)